# Import Library

## 
Files required 
1) gamma code = gamma(folder)
2) policy model = policy_model_discreteshift_final_3L_512H_s1_c3.pth
3) reference data = data(folder)
4) moving_average.py
5) GAMMA_obj_temp_depth.py

In [1]:
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
if torch.cuda.is_available():
    print("cuda is available")
else:
    print("cuda is NOT available")

import numpy as np
from tqdm import tqdm
import pickle
import seaborn as sns
import matplotlib.pyplot as plt
import time
import copy
from moving_average import moving_average_1d

from nn_functions import surrogate

import sys
sys.path.append('../1_model')
from TiDE import TideModule, quantile_loss  


cuda is available


## Import TiDE Model

In [2]:
import torch
import pickle

# Load model
with open('TiDE_params_single_track_square_MV_temp_depth_less_cov_0915_w50_p50.pkl', 'rb') as file:
    nominal_params = pickle.load(file)

TiDE = nominal_params['model'].to(device)
total_params = sum(p.numel() for p in TiDE.parameters())


In [3]:
import numpy as np
import pandas as pd
from typing import Optional, Tuple
import sys
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation
from IPython.display import HTML
from pickle import dump
from sklearn.preprocessing import MinMaxScaler
import time
from tqdm import tqdm
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
import os

# Resolve the primary compute device once and reuse it throughout the notebook.
device = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')

from moving_average import moving_average_1d
import copy

from GAMMA_obj_temp_depth import GAMMA_obj


Using device: cuda:0


# Import Policy Model

In [4]:
from policy import PolicyNN
import torch

# confirm which devices are available
print(torch.cuda.device_count())  # should be ≥1 to use CUDA

device = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')
# or force CPU: device = torch.device('cpu')

model = PolicyNN(
    past_input_dim=6,
    future_input_dim=6,
    output_dim=1,
    p=50,
    window=50,
    hidden_dim=512,
    n_layers=3,
    dropout_p=0.1
).to(device)

state = torch.load(
    "/home/ftk3187/github/DPC_research/02_DED/4_policy_0725/trainresults/policy_model_discreteshift_final_3L_512H_s1_c3.pth",
    map_location=device,
)
model.load_state_dict(state)
model.eval()

8


PolicyNN(
  (input_layer): Linear(in_features=600, out_features=512, bias=True)
  (input_ln): LayerNorm((512,), eps=1e-05, elementwise_affine=True)
  (dropout): Dropout(p=0.1, inplace=False)
  (hidden_layers): ModuleList(
    (0): Linear(in_features=512, out_features=512, bias=True)
  )
  (norm_layers): ModuleList(
    (0): LayerNorm((512,), eps=1e-05, elementwise_affine=True)
  )
  (output_layer): Linear(in_features=512, out_features=50, bias=True)
)

In [5]:
import math

class KFACLaplaceOnline:
    """Kronecker-factored Laplace approximation that updates online for PolicyNN."""

    def __init__(self, model: PolicyNN, prior_precision: float = 1.0, likelihood_std: float = 0.05, damping: float = 1e-3):
        self.model = model
        self.model.eval()
        self.prior_precision = prior_precision
        self.likelihood_std = likelihood_std
        self.damping = damping

        self.layers = [module for module in self.model.modules() if isinstance(module, nn.Linear)]
        if not self.layers:
            raise ValueError('PolicyNN must contain linear layers for KFAC Laplace.')

        self.activations = {layer: None for layer in self.layers}
        self.backprops = {layer: None for layer in self.layers}
        self.layer_stats = {
            layer: {
                'A': torch.zeros((layer.in_features, layer.in_features), dtype=torch.float64, device=layer.weight.device),
                'G': torch.zeros((layer.out_features, layer.out_features), dtype=torch.float64, device=layer.weight.device),
            }
            for layer in self.layers
        }
        self.layer_covariances = {}
        self.sample_count = 0
        self.output_count = 0

        self._register_hooks()

    def _register_hooks(self):
        def make_forward_hook(layer):
            def _hook(module, inputs, output):
                self.activations[layer] = inputs[0].detach()
            return _hook

        def make_backward_hook(layer):
            def _hook(module, grad_input, grad_output):
                self.backprops[layer] = grad_output[0].detach()
            return _hook

        for layer in self.layers:
            layer.register_forward_hook(make_forward_hook(layer))
            layer.register_full_backward_hook(make_backward_hook(layer))

    def _clear_backprops(self):
        for layer in self.layers:
            self.backprops[layer] = None

    def _update_covariances(self):
        if self.sample_count == 0 or self.output_count == 0:
            return
        for layer in self.layers:
            A_mean = self.layer_stats[layer]['A'] / self.sample_count
            G_mean = self.layer_stats[layer]['G'] / self.output_count

            in_dim = layer.in_features
            out_dim = layer.out_features

            eye_in = torch.eye(in_dim, dtype=torch.float64, device=A_mean.device)
            eye_out = torch.eye(out_dim, dtype=torch.float64, device=G_mean.device)

            A_damped = A_mean + (self.prior_precision + self.damping) * eye_in
            G_damped = G_mean + (self.prior_precision + self.damping) * eye_out

            self.layer_covariances[layer] = {
                'A': torch.linalg.inv(A_damped),
                'G': torch.linalg.inv(G_damped)
            }

    def evaluate(self, policy_past: torch.Tensor, policy_future: torch.Tensor, update_stats: bool = True):
        """Return mean control trajectory plus epistemic/total variance estimates."""
        self._clear_backprops()
        self.model.zero_grad(set_to_none=True)

        outputs = self.model((policy_past, policy_future))
        out_flat = outputs.view(outputs.shape[0], -1)
        batch_size, num_outputs = out_flat.shape

        mean = outputs.detach()
        epistemic_var = None
        total_var = None

        if self.layer_covariances:
            var_accum = torch.zeros(batch_size, num_outputs, dtype=torch.float64, device=outputs.device)
            for out_idx in range(num_outputs):
                grad_outputs = torch.zeros_like(out_flat)
                grad_outputs[:, out_idx] = 1.0
                self._clear_backprops()
                self.model.zero_grad(set_to_none=True)
                out_flat.backward(grad_outputs, retain_graph=True)

                for layer in self.layers:
                    a = self.activations[layer].to(torch.float64)
                    delta = self.backprops[layer].to(torch.float64)
                    A_cov = self.layer_covariances[layer]['A']
                    G_cov = self.layer_covariances[layer]['G']
                    delta_term = torch.einsum('bi,ij,bj->b', delta, G_cov, delta)
                    a_term = torch.einsum('bi,ij,bj->b', a, A_cov, a)
                    var_accum[:, out_idx] += delta_term * a_term

            epistemic_var = torch.clamp(var_accum.view_as(outputs).to(outputs.dtype), min=1e-12)
            total_var = epistemic_var + (self.likelihood_std ** 2)

        if update_stats:
            scale = 1.0 / (self.likelihood_std ** 2)
            scale_sqrt = math.sqrt(scale)

            with torch.no_grad():
                for layer in self.layers:
                    a = self.activations[layer].to(torch.float64)
                    self.layer_stats[layer]['A'] += a.transpose(0, 1) @ a

            for out_idx in range(num_outputs):
                grad_outputs = torch.zeros_like(out_flat)
                grad_outputs[:, out_idx] = scale_sqrt
                self._clear_backprops()
                self.model.zero_grad(set_to_none=True)
                retain = out_idx < (num_outputs - 1)
                out_flat.backward(grad_outputs, retain_graph=retain)

                for layer in self.layers:
                    delta = self.backprops[layer].to(torch.float64)
                    self.layer_stats[layer]['G'] += delta.transpose(0, 1) @ delta

            self.sample_count += batch_size
            self.output_count += batch_size * num_outputs
            self._update_covariances()

        self.model.zero_grad(set_to_none=True)
        self._clear_backprops()
        return mean, epistemic_var, total_var



In [6]:
kfac_laplace = KFACLaplaceOnline(model, prior_precision=1.0, likelihood_std=0.05, damping=1e-3)

uncertainty_log = {
    'mean_control': [],
    'epistemic_var_scaled': [],
    'epistemic_var_original': [],
    'total_var_scaled': [],
    'total_var_original': []
}


# Import Reference Data

In [7]:
import cupy as cp
device_id = min(0, cp.cuda.runtime.getDeviceCount() - 1)  # pick GPU 0 by default
cp.cuda.Device(device_id).use()

INPUT_DATA_DIR = "data"
SIM_DIR_NAME = "single_track_square"
BASE_LASER_FILE_DIR = "laser_power_profiles/csv"
CLOUD_TARGET_BASE_PATH = "result"
solidus_temp = 1600
window = 50
sim_interval = 5
init_runs = 50 #50 

GAMMA_class = GAMMA_obj(INPUT_DATA_DIR, SIM_DIR_NAME, BASE_LASER_FILE_DIR, CLOUD_TARGET_BASE_PATH, solidus_temp, window, init_runs, sim_interval, laser_power_number=1)
init_avg = GAMMA_class.run_initial_steps()
init_avg = torch.tensor(init_avg,dtype=torch.float32)[:,-window:] # shape = [2,50]

100%|██████████| 250/250 [00:06<00:00, 41.48it/s]


In [8]:
df_one_print = pd.read_csv('single_track_ref.csv')

loc_X_list = df_one_print["X"].to_numpy().reshape(-1,1)
loc_Y_list = df_one_print["Y"].to_numpy().reshape(-1,1)
loc_Z_list = df_one_print["Z"].to_numpy().reshape(-1,1)
dist_X_list = df_one_print["Dist_to_nearest_X"].to_numpy().reshape(-1,1)
dist_Y_list = df_one_print["Dist_to_nearest_Y"].to_numpy().reshape(-1,1)
scan_spd_list = df_one_print["scanning_speed"].to_numpy().reshape(-1,1)

# laser power
laser_power_ref = torch.tensor(df_one_print["Laser_power"].to_numpy().reshape(-1,1),dtype=torch.float32)
laser_power_past = laser_power_ref[:window]
fix_covariates = torch.tensor(np.concatenate((loc_Z_list,dist_X_list,dist_Y_list),axis=1),dtype=torch.float32)

# apply moving average for mp temp
mp_temp_raw = df_one_print["melt_pool_temperature"].to_numpy()
mp_temp_mv = moving_average_1d(mp_temp_raw,4)
mp_temp = copy.deepcopy(mp_temp_raw)
mp_temp[1:-2] = mp_temp_mv
mp_temp = mp_temp

mp_temp_ref = torch.tensor(mp_temp,dtype=torch.float32)

x_min = torch.tensor([[0.0, 0.75, 0.75, 504.26]], dtype=torch.float32).to(device)
x_max = torch.tensor([[7.5, 20.0, 20.0, 732.298]], dtype=torch.float32).to(device)

y_min = torch.tensor([[436.608, -0.559]], dtype=torch.float32).to(device)
y_max = torch.tensor([[4509.855, 0.551]], dtype=torch.float32).to(device)

# Precompute constants that map control values between scaled and original units.
LASER_SCALE = float(0.5 * (x_max[0, 3].item() - x_min[0, 3].item()))
LASER_OFFSET = float(x_min[0, 3].item())


In [9]:
x_min = torch.tensor([[0.0, 0.75, 0.75, 504.26]], dtype=torch.float32).to(device)
x_max = torch.tensor([[7.5, 20.0, 20.0, 732.298]], dtype=torch.float32).to(device)

y_min = torch.tensor([[436.608, -0.559]], dtype=torch.float32).to(device)
y_max = torch.tensor([[4509.855, 0.551]], dtype=torch.float32).to(device)


In [10]:
def normalize_x(x, dim_id):
    x_min_selected = x_min[0, dim_id]
    x_max_selected = x_max[0, dim_id]
    return 2 * (x - x_min_selected) / (x_max_selected - x_min_selected) - 1

def inverse_normalize_x(x_norm, dim_id):
    x_min_selected = x_min[0, dim_id]
    x_max_selected = x_max[0, dim_id]
    return 0.5 * (x_norm + 1) * (x_max_selected - x_min_selected) + x_min_selected

def normalize_y(y, dim_id):
    y_min_selected = y_min[0, dim_id]
    y_max_selected = y_max[0, dim_id]
    return 2 * (y - y_min_selected) / (y_max_selected - y_min_selected) - 1

def inverse_normalize_y(y_norm, dim_id):
    y_min_selected = y_min[0, dim_id]
    y_max_selected = y_max[0, dim_id]
    return 0.5 * (y_norm + 1) * (y_max_selected - y_min_selected) + y_min_selected


# Sub-Function ; Run Policy

In [11]:
import os
import numpy as np
import torch
from pathlib import Path

# ============================================================
# 🔧 Utility Functions
# ============================================================

def clone_gamma(G):
    import copy
    return copy.deepcopy(G)

def rollout_future(G_clone, control_seq):
    temps, depths = [], []
    for u in control_seq:
        x, d = G_clone.run_sim_interval(float(u))
        temps.append(x)
        depths.append(d)
    return temps, depths


# ============================================================
# 🔧 TiDE 보조 함수: normalized → physical
# ============================================================

def tide_to_physical(y_norm, quant_idx=1):
    """ TiDE 출력 → 물리 단위 (정규화 해제) """
    y_med = y_norm[..., quant_idx]  # (1, P, 2)
    temp_norm  = y_med[:, :, 0]
    depth_norm = y_med[:, :, 1]
    temp_phys  = inverse_normalize_y(temp_norm,  dim_id=[0])
    depth_phys = inverse_normalize_y(depth_norm, dim_id=[1])
    return (
        temp_phys.squeeze(0).detach().cpu().numpy().tolist(),
        depth_phys.squeeze(0).detach().cpu().numpy().tolist(),
    )


# ============================================================
# 🚀 Main Function: run_one_step_policy (Epistemic + Aleatoric)
# ============================================================

device = next(TiDE.parameters()).device if TiDE is not None else next(model.parameters()).device

def run_one_step_policy(
    GAMMA_obj,
    policy_model,
    P,
    window,
    laplace=None,
    uncertainty_log=None,
    tide_model=None,
    tide_quantile_idx=1,
    rollout_interval=50,
    do_save_plot=True
):
    """
    한 스텝 실행:
      - Laplace 기반 policy variance → epistemic & aleatoric 밴드 계산
      - TiDE surrogate로 미래 temp/depth 예측 (두 밴드 각각)
      - 플롯 시 두 불확실성 효과를 시각적으로 분리 가능
    """

    # ------------------------------------------------------------
    # 1️⃣ 입력 구성
    # ------------------------------------------------------------
    mp_temp_ref = GAMMA_obj.ref[GAMMA_obj.MPC_counter : GAMMA_obj.MPC_counter + P]
    mp_temp_ref_t = torch.as_tensor(mp_temp_ref, dtype=torch.float32, device=device).reshape(1, P, 1)

    mp_temp_past_t = GAMMA_obj.x_past.T.unsqueeze(0).to(device)
    laser_past_t   = GAMMA_obj.u_past.view(1, -1, 1).to(device)

    fix_cov_past   = GAMMA_obj.fix_cov_all[GAMMA_obj.MPC_counter - window : GAMMA_obj.MPC_counter, :]
    fix_cov_past_t = torch.as_tensor(fix_cov_past, dtype=torch.float32, device=device).unsqueeze(0)

    fix_cov_past_s = normalize_x(fix_cov_past_t, dim_id=[0,1,2])
    laser_past_s   = normalize_x(laser_past_t,   dim_id=[3])
    mp_temp_past_s = normalize_y(mp_temp_past_t, dim_id=[0,1])

    policy_in_past = torch.cat((fix_cov_past_s, laser_past_s, mp_temp_past_s), dim=2)

    fix_cov_future   = GAMMA_obj.fix_cov_all[GAMMA_obj.MPC_counter : GAMMA_obj.MPC_counter + P, :]
    fix_cov_future_t = torch.as_tensor(fix_cov_future, dtype=torch.float32, device=device).unsqueeze(0)
    fix_cov_future_s = normalize_x(fix_cov_future_t, dim_id=[0,1,2])

    mp_temp_ref_s = normalize_y(mp_temp_ref_t, dim_id=[0])[:, :, 0].unsqueeze(-1)

    depth_lower_const = 0.1423
    depth_upper_const = 0.4126
    y_const_s = torch.tensor([[depth_lower_const, depth_upper_const]] * P,
                             dtype=torch.float32, device=device).reshape(1, P, 2)

    policy_in_future = torch.cat((fix_cov_future_s, mp_temp_ref_s, y_const_s), dim=2)

    # ------------------------------------------------------------
    # 2️⃣ 정책 예측
    # ------------------------------------------------------------
    if laplace is not None:
        u_pred, epistemic_var, total_var = laplace.evaluate(policy_in_past, policy_in_future, update_stats=True)
    else:
        u_pred = policy_model((policy_in_past, policy_in_future))
        epistemic_var = None
        total_var     = None

    # ------------------------------------------------------------
    # 3️⃣ 로그 기록
    # ------------------------------------------------------------
    laser_span   = (x_max[0, 3] - x_min[0, 3]).item()
    laser_scale  = 0.5 * laser_span
    laser_offset = x_min[0, 3].item()

    if uncertainty_log is not None:
        control_scaled_0 = u_pred[0, 0, 0].detach().cpu()
        control_original = float((control_scaled_0 + 1.0) * laser_scale + laser_offset)
        uncertainty_log['mean_control'].append(control_original)

        if (epistemic_var is not None) and (total_var is not None):
            var_e = float(epistemic_var[0, 0, 0].detach().cpu())
            var_t = float(total_var[0, 0, 0].detach().cpu())
            uncertainty_log['epistemic_var_scaled'].append(var_e)
            uncertainty_log['total_var_scaled'].append(var_t)
            uncertainty_log['epistemic_var_original'].append((laser_scale**2) * var_e)
            uncertainty_log['total_var_original'].append((laser_scale**2) * var_t)
        else:
            for k in ['epistemic_var_scaled','total_var_scaled','epistemic_var_original','total_var_original']:
                uncertainty_log[k].append(None)

    # ------------------------------------------------------------
    # 4️⃣ 미래 control band 계산 (Epistemic / Aleatoric)
    # ------------------------------------------------------------
    k_before = GAMMA_obj.MPC_counter
    mean_scaled_h = u_pred[0, :, 0].detach().cpu().numpy()

    if (epistemic_var is not None) and (total_var is not None):
        epi_h  = epistemic_var[0, :, 0].detach().cpu().numpy()
        tot_h  = total_var[0, :, 0].detach().cpu().numpy()
        alea_h = np.maximum(tot_h - epi_h, 0.0)

        std_epistemic = np.sqrt(epi_h)
        std_aleatoric = np.sqrt(alea_h)
        std_total     = np.sqrt(tot_h)

        upper_epi = mean_scaled_h + 1.28 * std_epistemic
        lower_epi = mean_scaled_h - 1.28 * std_epistemic
        upper_alea = mean_scaled_h + 1.28 * std_aleatoric
        lower_alea = mean_scaled_h - 1.28 * std_aleatoric

    else:
        upper_epi = lower_epi = upper_alea = lower_alea = mean_scaled_h.copy()

    mean_phys  = (mean_scaled_h  + 1.0) * laser_scale + laser_offset
    upper_epi_phys  = (upper_epi  + 1.0) * laser_scale + laser_offset
    lower_epi_phys  = (lower_epi  + 1.0) * laser_scale + laser_offset
    upper_alea_phys = (upper_alea + 1.0) * laser_scale + laser_offset
    lower_alea_phys = (lower_alea + 1.0) * laser_scale + laser_offset

    # ------------------------------------------------------------
    # 5️⃣ TiDE surrogate 미래 예측 (Epistemic & Aleatoric 모두)
    # ------------------------------------------------------------
    if (tide_model is not None) and ((k_before + 1) % rollout_interval == 0):
        device_tide = next(tide_model.parameters()).device
        tide_model.eval()

        with torch.no_grad():
            x_past_s   = torch.cat((fix_cov_past_s, laser_past_s), dim=2).to(device_tide)
            past_cov_s = torch.cat((mp_temp_past_s, x_past_s),     dim=2).to(device_tide)

            def run_tide(upper_scaled, lower_scaled):
                up_t = torch.as_tensor(upper_scaled, dtype=torch.float32, device=device_tide).view(1, P, 1)
                low_t = torch.as_tensor(lower_scaled, dtype=torch.float32, device=device_tide).view(1, P, 1)
                x_future_up  = torch.cat((fix_cov_future_s.to(device_tide), up_t),  dim=2)
                x_future_low = torch.cat((fix_cov_future_s.to(device_tide), low_t), dim=2)
                y_up  = tide_model((past_cov_s, x_future_up,  None))
                y_low = tide_model((past_cov_s, x_future_low, None))
                return tide_to_physical(y_up,  tide_quantile_idx), tide_to_physical(y_low, tide_quantile_idx)

            # --- epistemic rollout ---
            (temp_up_e, depth_up_e), (temp_low_e, depth_low_e) = run_tide(upper_epi, lower_epi)
            # --- aleatoric rollout ---
            (temp_up_a, depth_up_a), (temp_low_a, depth_low_a) = run_tide(upper_alea, lower_alea)

        # ✅ GAMMA_obj에 저장
        GAMMA_obj.future_temp_upper_epistemic  = temp_up_e
        GAMMA_obj.future_temp_lower_epistemic  = temp_low_e
        GAMMA_obj.future_depth_upper_epistemic = depth_up_e
        GAMMA_obj.future_depth_lower_epistemic = depth_low_e

        GAMMA_obj.future_temp_upper_aleatoric  = temp_up_a
        GAMMA_obj.future_temp_lower_aleatoric  = temp_low_a
        GAMMA_obj.future_depth_upper_aleatoric = depth_up_a
        GAMMA_obj.future_depth_lower_aleatoric = depth_low_a

        GAMMA_obj.future_control_mean  = mean_phys
        GAMMA_obj.future_control_upper_epistemic  = upper_epi_phys
        GAMMA_obj.future_control_lower_epistemic  = lower_epi_phys
        GAMMA_obj.future_control_upper_aleatoric  = upper_alea_phys
        GAMMA_obj.future_control_lower_aleatoric  = lower_alea_phys

        if do_save_plot:
            save_dir = Path("plots"); save_dir.mkdir(parents=True, exist_ok=True)
            fig_path = save_dir / f"mpc_step_{k_before:04d}_epi_alea.png"
            plot_fig(MPC_GAMMA=GAMMA_obj, i=k_before, uncertainty_log=uncertainty_log)
            plt.savefig(fig_path, dpi=200); plt.close()
            print(f"[INFO] Saved TiDE rollout plot with epistemic/aleatoric bands at k={k_before}: {fig_path.resolve()}")

    # ------------------------------------------------------------
    # 6️⃣ 실제 제어 적용
    # ------------------------------------------------------------
    u_first   = u_pred[0, 0]
    u_applied = float(inverse_normalize_x(u_first, dim_id=[3]))
    x_current, depth_current = GAMMA_obj.run_sim_interval(u_applied)

    GAMMA_obj.x_past[:, :-1] = GAMMA_obj.x_past[:, 1:]
    GAMMA_obj.x_past[0, -1]  = x_current
    GAMMA_obj.x_past[1, -1]  = depth_current
    GAMMA_obj.u_past[:-1]    = GAMMA_obj.u_past[1:].clone()
    GAMMA_obj.u_past[-1]     = u_applied

    GAMMA_obj.x_hat_current  = torch.tensor([x_current, depth_current], device=device)
    GAMMA_obj.x_sys_current  = torch.tensor([[x_current], [depth_current]], device=device)

    GAMMA_obj.MPC_counter += 1

    new_state = torch.tensor([[x_current, depth_current]], device=GAMMA_obj.x_past_save.device)
    GAMMA_obj.x_past_save = torch.cat((GAMMA_obj.x_past_save, new_state), dim=0)
    new_u = torch.tensor([[u_applied]], device=GAMMA_obj.u_past_save.device)
    GAMMA_obj.u_past_save = torch.cat((GAMMA_obj.u_past_save, new_u), dim=0)


# Sub-Function ; Plot Rollout

In [12]:
def plot_fig(MPC_GAMMA, i, uncertainty_log=None, horizon=50):
    import numpy as np
    import matplotlib.pyplot as plt

    plt.figure(figsize=[12, 10])

    past_len = len(MPC_GAMMA.x_past_save)
    t_past = np.arange(past_len)
    t_future = np.arange(i, i + horizon)

    # ===================== TEMPERATURE =====================
    plt.subplot(3, 1, 1)

    # ---- Future: Epistemic band ----
    if hasattr(MPC_GAMMA, "future_temp_upper_epistemic"):
        fut_h = min(horizon, len(MPC_GAMMA.future_temp_upper_epistemic))
        t_f = t_future[:fut_h]
        plt.fill_between(t_f,
                         MPC_GAMMA.future_temp_lower_epistemic[:fut_h],
                         MPC_GAMMA.future_temp_upper_epistemic[:fut_h],
                         color='tab:blue', alpha=0.25, label='Epistemic band')
    # ---- Future: Aleatoric band ----
    if hasattr(MPC_GAMMA, "future_temp_upper_aleatoric"):
        fut_h = min(horizon, len(MPC_GAMMA.future_temp_upper_aleatoric))
        t_f = t_future[:fut_h]
        plt.fill_between(t_f,
                         MPC_GAMMA.future_temp_lower_aleatoric[:fut_h],
                         MPC_GAMMA.future_temp_upper_aleatoric[:fut_h],
                         color='tab:orange', alpha=0.25, label='Aleatoric band')

    # ---- (fallback) Generic future band ----
    elif hasattr(MPC_GAMMA, "future_temp_upper"):
        fut_h = min(horizon, len(MPC_GAMMA.future_temp_upper))
        plt.plot(t_future[:fut_h], MPC_GAMMA.future_temp_upper[:fut_h], 'r--', label="Future upper")
        plt.plot(t_future[:fut_h], MPC_GAMMA.future_temp_lower[:fut_h], 'g--', label="Future lower")

    # ---- Past trajectory + ref ----
    plt.plot(t_past, MPC_GAMMA.x_past_save[:, 0], color="blue", label="GAMMA simulation")
    if hasattr(MPC_GAMMA, "ref"):
        fut_h_ref = min(horizon, len(MPC_GAMMA.ref) - i)
        plt.plot(t_past, MPC_GAMMA.ref[:past_len], color="orange", label="Reference")
        if fut_h_ref > 0:
            plt.plot(t_future[:fut_h_ref], MPC_GAMMA.ref[i:i+fut_h_ref],
                     color="orange", alpha=0.8, linestyle='--', label="Future ref")

    plt.ylabel("MP Temp (K)")
    plt.title("MPC Temperature – Epistemic & Aleatoric Uncertainty")
    plt.legend(loc="best", fontsize=9)
    plt.xlim(i - 100, i + 60)

    # ===================== DEPTH =====================
    plt.subplot(3, 1, 2)

    # ---- Future: Epistemic band ----
    if hasattr(MPC_GAMMA, "future_depth_upper_epistemic"):
        fut_h = min(horizon, len(MPC_GAMMA.future_depth_upper_epistemic))
        t_f = t_future[:fut_h]
        plt.fill_between(t_f,
                         MPC_GAMMA.future_depth_lower_epistemic[:fut_h],
                         MPC_GAMMA.future_depth_upper_epistemic[:fut_h],
                         color='tab:blue', alpha=0.25, label='Epistemic band')
    # ---- Future: Aleatoric band ----
    if hasattr(MPC_GAMMA, "future_depth_upper_aleatoric"):
        fut_h = min(horizon, len(MPC_GAMMA.future_depth_upper_aleatoric))
        t_f = t_future[:fut_h]
        plt.fill_between(t_f,
                         MPC_GAMMA.future_depth_lower_aleatoric[:fut_h],
                         MPC_GAMMA.future_depth_upper_aleatoric[:fut_h],
                         color='tab:orange', alpha=0.25, label='Aleatoric band')

    # ---- (fallback) Generic ----
    elif hasattr(MPC_GAMMA, "future_depth_upper"):
        fut_h = min(horizon, len(MPC_GAMMA.future_depth_upper))
        plt.plot(t_future[:fut_h], MPC_GAMMA.future_depth_upper[:fut_h], 'r--', label="Future upper")
        plt.plot(t_future[:fut_h], MPC_GAMMA.future_depth_lower[:fut_h], 'g--', label="Future lower")

    # ---- Past & constraints ----
    plt.plot(t_past, MPC_GAMMA.x_past_save[:, 1], color="blue", label="GAMMA simulation")
    UB, LB = 0.225, 0.075
    plt.plot(t_past, UB * np.ones(past_len), color="red", linestyle='--', label="UB")
    plt.plot(t_past, LB * np.ones(past_len), color="green", linestyle='--', label="LB")
    plt.plot(t_future, UB * np.ones(horizon), color="red", alpha=0.5)
    plt.plot(t_future, LB * np.ones(horizon), color="green", alpha=0.5)

    plt.ylabel("MP Depth (mm)")
    plt.title("MPC Depth – Epistemic & Aleatoric Uncertainty")
    plt.legend(loc="best", fontsize=9)
    plt.xlim(i - 100, i + 60)

    # ===================== LASER POWER =====================
    plt.subplot(3, 1, 3)

    # ---- Future: Epistemic ----
    if hasattr(MPC_GAMMA, "future_control_upper_epistemic"):
        fut_h = min(horizon, len(MPC_GAMMA.future_control_upper_epistemic))
        t_f = t_future[:fut_h]
        plt.fill_between(t_f,
                         MPC_GAMMA.future_control_lower_epistemic[:fut_h],
                         MPC_GAMMA.future_control_upper_epistemic[:fut_h],
                         color='tab:blue', alpha=0.25, label="Epistemic band (future)")

    # ---- Future: Aleatoric ----
    if hasattr(MPC_GAMMA, "future_control_upper_aleatoric"):
        fut_h = min(horizon, len(MPC_GAMMA.future_control_upper_aleatoric))
        t_f = t_future[:fut_h]
        plt.fill_between(t_f,
                         MPC_GAMMA.future_control_lower_aleatoric[:fut_h],
                         MPC_GAMMA.future_control_upper_aleatoric[:fut_h],
                         color='tab:orange', alpha=0.25, label="Aleatoric band (future)")

    # ---- Fallback: Generic ----
    elif hasattr(MPC_GAMMA, "future_control_mean"):
        fut_h = min(horizon, len(MPC_GAMMA.future_control_mean))
        t_f = t_future[:fut_h]
        plt.fill_between(t_f,
                         MPC_GAMMA.future_control_lower[:fut_h],
                         MPC_GAMMA.future_control_upper[:fut_h],
                         alpha=0.3, color='gray', label="Uncertainty band (future)")
        plt.plot(t_f, MPC_GAMMA.future_control_mean[:fut_h], 'k--', label="Future control mean")

    # ---- Past applied control ----
    plt.plot(t_past, MPC_GAMMA.u_past_save[:past_len], color="blue", label="Laser power (applied)")

    plt.ylabel("Laser Power (W)")
    plt.xlabel("MPC time step (0.0355 sec/iteration)")
    plt.title("Laser Power – Epistemic & Aleatoric Uncertainty")
    plt.legend(loc="best", fontsize=9)
    plt.xlim(i - 100, i + 60)

    plt.tight_layout()
    return plt


# Execution

In [13]:
print("=== DEVICE CHECK ===")
print("TiDE model:", next(TiDE.parameters()).device)
print("policy_model:", next(model.parameters()).device)
print("GAMMA_class type:", type(GAMMA_class))

if hasattr(GAMMA_class, "x_past"):
    print("GAMMA_class.x_past device:", GAMMA_class.x_past.device)
else:
    print("GAMMA_class.x_past not found")


=== DEVICE CHECK ===
TiDE model: cuda:0
policy_model: cuda:0
GAMMA_class type: <class 'GAMMA_obj_temp_depth.GAMMA_obj'>
GAMMA_class.x_past not found


In [14]:
# step #
P = 50
N_step = len(mp_temp_ref) - init_runs + 50

# initialize GAMMA class
GAMMA_class.ref = mp_temp_ref
GAMMA_class.fix_cov_all = fix_covariates
GAMMA_class.x_past = init_avg.clone()
GAMMA_class.u_past = laser_power_past.clone()

GAMMA_class.x_hat_current = GAMMA_class.x_past[:, -1]
GAMMA_class.x_sys_current = GAMMA_class.x_past[:, -1].reshape(2, 1)

GAMMA_class.x_past_save = GAMMA_class.x_past.T.clone()
GAMMA_class.u_past_save = GAMMA_class.u_past.clone()
GAMMA_class.MPC_counter = window


# execution loop
from tqdm import tqdm

for i in tqdm(range(N_step)):
    run_one_step_policy(
        GAMMA_class,
        model,
        P=P,
        window=window,
        laplace=kfac_laplace,
        uncertainty_log=uncertainty_log,
        tide_model=TiDE,            # ✅ TiDE 모델 전달
        tide_quantile_idx=1,        # ✅ 중앙값 (50%) 예측 사용
        rollout_interval=100,        # ✅ 50 스텝마다만 TiDE 롤아웃
        do_save_plot=True           # ✅ TiDE 기반 plot 자동 저장
    )

    # ✅ 별도 plot_fig 호출 제거 (이미 run_one_step_policy 내부에서 50 step마다 저장)
    # ❌ plot_fig(GAMMA_class, i)


  1%|          | 50/6295 [00:15<48:29,  2.15it/s]

[INFO] Saved TiDE rollout plot with epistemic/aleatoric bands at k=99: /home/ftk3187/github/DPC_research/02_DED/4_policy_0725/plots/mpc_step_0099_epi_alea.png


  2%|▏         | 150/6295 [00:45<46:29,  2.20it/s]

[INFO] Saved TiDE rollout plot with epistemic/aleatoric bands at k=199: /home/ftk3187/github/DPC_research/02_DED/4_policy_0725/plots/mpc_step_0199_epi_alea.png


  4%|▍         | 250/6295 [01:15<45:34,  2.21it/s]

...

[INFO] Saved TiDE rollout plot with epistemic/aleatoric bands at k=6099: /home/ftk3187/github/DPC_research/02_DED/4_policy_0725/plots/mpc_step_6099_epi_alea.png


 98%|█████████▊| 6150/6295 [32:56<01:08,  2.13it/s]

[INFO] Saved TiDE rollout plot with epistemic/aleatoric bands at k=6199: /home/ftk3187/github/DPC_research/02_DED/4_policy_0725/plots/mpc_step_6199_epi_alea.png


 98%|█████████▊| 6193/6295 [33:09<00:32,  3.11it/s]


_LinAlgError: linalg.inv: The diagonal element 598 is zero, the inversion could not be completed because the input matrix is singular.

# Inspect KFAC Uncertainty


In [ ]:
uncertainty_df = pd.DataFrame(uncertainty_log)
uncertainty_df.head()
